# Qwen Regional Grounding

This notebook generates `instructions_200.json`, which is used by the downstream regional style-transfer notebook.

The notebook loads the required data and Qwen model, finds the corresponding image for each sample, and generates a regional style instruction based on the provided design information.

### Main steps

1. **Google Drive storage**
   The working directory is set to Google Drive so that the generated files and checkpoints are retained after the Colab session ends.

2. **Image discovery**
   The notebook searches for the image associated with each sample. Category folder names are matched without considering letter case, and different filename patterns are checked when locating an image.

3. **Pre-flight check**
   Before starting model inference, the notebook checks that the required images are available.

4. **Qwen instruction generation**
   Qwen generates a regional style instruction using the information provided in the prompt. Greedy decoding is used for consistent generation.

5. **Output validation**
   The generated response is processed and checked to make sure that it follows the expected JSON structure and contains the required regional information.

6. **Checkpointing**
   Progress is saved every 10 samples. If the Colab runtime is interrupted, the saved checkpoint can be used to continue processing without starting again from the beginning.

7. **Saving the results**
   The final instructions and related information are saved to Google Drive. A manifest is also created to keep track of the generated instructions and their corresponding samples.

The resulting `instructions_200.json` file is used as input for the regional style-transfer stage.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, sys, re, json, glob, time, random, hashlib, subprocess
from datetime import datetime, timezone

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

    # Keep the Colab session active while the model is running.
    from IPython.display import display, Javascript
    display(Javascript('''
        function KeepAlive(){
            document.querySelector("colab-toolbar-button#connect")?.click();
        }
        setInterval(KeepAlive, 60000);
    '''))
    print('Drive mounted, keepalive installed')
else:
    print('not running on Colab: skipping Drive mount and keepalive')

RUN_ID = datetime.now(timezone.utc).strftime('qwen_run_%Y%m%dT%H%M%SZ')

CONFIG = {
    # Dataset settings used by the regional style-transfer notebook.
    'dataset_root': '/content/drive/MyDrive',
    'categories': ('City', 'Landscape', 'Nature', 'Sunset', 'Winter'),
    'existing_instructions_path': '/content/drive/MyDrive/instructions_200.json',

    # Set this to a number when testing with only a smaller number of samples.
    'limit_samples': None,

    # Styles used in the downstream notebook.
    'allowed_styles': ['Van Gogh', 'Cezanne', 'Monet', 'Kandinsky', 'Delaunay'],
    'style_name_aliases': {'Picasso': 'Delaunay'},

    # Qwen model settings.
    'qwen_model': 'Qwen/Qwen2-VL-2B-Instruct', 'qwen_load_in_4bit': False,
    # 'qwen_model': 'Qwen/Qwen2-VL-7B-Instruct', 'qwen_load_in_4bit': True,
    'qwen_dtype': 'auto',
    'qwen_max_tokens': 300,
    'qwen_max_pixels': 512 * 512,

    # Store the working files in Google Drive so they are available after the session.
    'work_dir': '/content/drive/MyDrive/qwen_grounding',
}

for sub in ('raw', 'manifest'):
    os.makedirs(os.path.join(CONFIG['work_dir'], sub), exist_ok=True)
RAW_DIR = os.path.join(CONFIG['work_dir'], 'raw')
MANIFEST_DIR = os.path.join(CONFIG['work_dir'], 'manifest')
CHECKPOINT_PATH = os.path.join(MANIFEST_DIR, 'checkpoint.json')


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def sha256_text(s):
    return hashlib.sha256(s.encode()).hexdigest()


print('RUN_ID:', RUN_ID)
print('work_dir (Drive-backed):', CONFIG['work_dir'])
print(f"model: {CONFIG['qwen_model']}  4bit={CONFIG['qwen_load_in_4bit']}")
print(f"samples: {'ALL 200' if CONFIG['limit_samples'] is None else CONFIG['limit_samples']}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

Drive mounted, keepalive installed
RUN_ID: qwen_run_20260827T073517Z
work_dir (Drive-backed): /content/drive/MyDrive/qwen_grounding
model: Qwen/Qwen2-VL-2B-Instruct  4bit=False
samples: ALL 200


---
## Dependencies and model loading

In [ ]:
import torch
HAS_CUDA = torch.cuda.is_available()
print('CUDA:', HAS_CUDA)
if HAS_CUDA:
    print(f'VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.45', 'accelerate', 'qwen-vl-utils', 'bitsandbytes'],
               check=True)
print('dependencies installed')

CUDA: True
VRAM free: 15.53 GB
dependencies installed


In [ ]:
import gc

MODELS = {}


def load_qwen():
    if 'model' in MODELS:
        return MODELS['model'], MODELS['processor']
    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

    kwargs = dict(torch_dtype='auto' if CONFIG['qwen_dtype'] == 'auto'
                  else getattr(torch, CONFIG['qwen_dtype']), device_map='auto')
    if CONFIG['qwen_load_in_4bit']:
        from transformers import BitsAndBytesConfig
        kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)

    model = Qwen2VLForConditionalGeneration.from_pretrained(CONFIG['qwen_model'], **kwargs)
    processor = AutoProcessor.from_pretrained(
        CONFIG['qwen_model'], min_pixels=256 * 28 * 28, max_pixels=CONFIG['qwen_max_pixels'])
    MODELS['model'], MODELS['processor'] = model, processor
    print(f"loaded {CONFIG['qwen_model']}, 4bit={CONFIG['qwen_load_in_4bit']}")
    if HAS_CUDA:
        print(f'VRAM free after load: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB')
    return model, processor


def unload_qwen():
    for k in ('model', 'processor'):
        MODELS.pop(k, None)
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()
    print('Qwen unloaded' + (f', VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB'
                             if HAS_CUDA else ''))

---
## Design table

In [ ]:
def parse_instruction(instruction):

    '''Extract the region and style pairs from an instruction.

    The same format is used in the downstream style-transfer notebook.
    This is used to create the design table.'''

    pairs = []

    for region, style in re.findall(
            r'(?:the\s+)?([a-zA-Z ]+?)\s+like\s+([A-Za-z_ ]+?)(?:\s+and\s+|\s*,\s*|$)',
            instruction, re.IGNORECASE):

        region = re.sub(r'^(style|make|render|paint)\s+', '', region.strip(), flags=re.I).strip()
        region = re.sub(r'^the\s+', '', region, flags=re.I).strip()
        style = style.strip().replace('_', ' ')

        if region and style:
            pairs.append({'region': region, 'style': style})

    return pairs


def load_design_table(path):

    with open(path) as f:
        samples = json.load(f)

    rows = []

    for s in samples:
        pairs = parse_instruction(s['instruction'])
        by_region = {p['region']: p['style'] for p in pairs}

        r1, r2 = s['regions'][0], s['regions'][1]
        s1 = CONFIG['style_name_aliases'].get(by_region.get(r1), by_region.get(r1))
        s2 = CONFIG['style_name_aliases'].get(by_region.get(r2), by_region.get(r2))

        rows.append({'sample_id': s['id'], 'region1': r1, 'region2': r2,
                     'style1': s1, 'style2': s2})

    return rows


def validate_design_table(rows):
    problems = []
    ids = [r['sample_id'] for r in rows]
    if len(set(ids)) != len(ids):
        problems.append(f'duplicate sample_id(s): {sorted({i for i in ids if ids.count(i) > 1})}')
    allowed = set(CONFIG['allowed_styles']) | set(CONFIG['style_name_aliases'])
    for r in rows:
        for f in ('style1', 'style2'):
            if r[f] not in allowed:
                problems.append(f"{r['sample_id']}: {f}='{r[f]}' not in {sorted(allowed)}")
        if r['region1'] == r['region2']:
            problems.append(f"{r['sample_id']}: region1 == region2 ('{r['region1']}')")

    return problems
design_rows = load_design_table(CONFIG['existing_instructions_path'])
problems = validate_design_table(design_rows)
if problems:
    raise RuntimeError('design table problems:\n' + '\n'.join(f'  {p}' for p in problems))
DESIGN_BY_ID = {r['sample_id']: r for r in design_rows}
SAMPLE_IDS = sorted(DESIGN_BY_ID)
if CONFIG['limit_samples']:
    SAMPLE_IDS = SAMPLE_IDS[:CONFIG['limit_samples']]

print(f'{len(SAMPLE_IDS)} samples selected from a design table of {len(design_rows)}')

200 samples selected from a design table of 200


## Dataset discovery

The notebook checks the dataset folders and finds the image corresponding to each sample.

- **Category folders:** Folder names are matched without considering letter case, and any extra spaces at the end of the folder name are ignored.

- **Image files:** The search first checks for filenames that match the sample ID. If no match is found, it also checks the filename without considering letter case and then looks for the sample ID anywhere in the filename.

If an image cannot be found, the notebook prints the sample ID and shows the files available in the corresponding folder. This helps to check whether the image is missing or has a different filename.

In [ ]:
def _discover_category_folders(root, categories):
    folders = {}
    entries = os.listdir(root)
    # Check for folders with the expected category names first.
    for entry in entries:
        full = os.path.join(root, entry)
        if os.path.isdir(full) and entry in categories:
            folders[entry[0].upper()] = full
    # If needed, also check folder names without case and extra spaces.
    missing = [c for c in categories if c[0].upper() not in folders]
    if missing:
        norm = {c.strip().lower(): c for c in categories}
        for entry in entries:
            full = os.path.join(root, entry)
            key = entry.strip().lower()
            if os.path.isdir(full) and key in norm and norm[key] in missing:
                cat = norm[key]
                folders[cat[0].upper()] = full
                print(f'  note: matched folder "{entry}" to category "{cat}" '
                     f'(case/whitespace-insensitive fallback)')
    still_missing = [c for c in categories if c[0].upper() not in folders]
    assert not still_missing, (
        f'category folders not found even with case-insensitive matching: {still_missing}. '
        f'Folders present at {root}: {sorted(e for e in entries if os.path.isdir(os.path.join(root, e)))}')
    return folders


def find_content_image(sample_id, category_folders):
    prefix = sample_id[0].upper()
    assert prefix in category_folders, f'no folder discovered for prefix "{prefix}"'
    folder = category_folders[prefix]

    # First try the expected filename patterns.
    matches = sorted(glob.glob(os.path.join(folder, f'{sample_id}_*')))
    matches += sorted(glob.glob(os.path.join(folder, f'{sample_id}.*')))
    if matches:
        return matches[0]

    # If that does not match, try the same patterns without case sensitivity.
    all_files = os.listdir(folder)
    ci_matches = sorted(f for f in all_files
                        if re.match(rf'^{re.escape(sample_id)}[._]', f, re.IGNORECASE))
    if ci_matches:
        return os.path.join(folder, ci_matches[0])

    # As a final check, look for the sample ID anywhere in the filename.
    substr_matches = sorted(f for f in all_files if sample_id.lower() in f.lower())
    if substr_matches:
        return os.path.join(folder, substr_matches[0])

    # Show the available files if no matching image is found.
    raise FileNotFoundError(
        f'no content image for {sample_id} in {folder} (tried exact, case-insensitive '
        f'prefix, and substring matching). {len(all_files)} file(s) present, first 10: '
        f'{sorted(all_files)[:10]}')


PREFIX_TO_CATEGORY = {c[0].upper(): c for c in CONFIG['categories']}
category_folders = _discover_category_folders(CONFIG['dataset_root'], CONFIG['categories'])
print('dataset discovery ready (case/whitespace-tolerant)')
print('folders:', {k: os.path.basename(v) for k, v in category_folders.items()})

dataset discovery ready (case/whitespace-tolerant)
folders: {'W': 'Winter', 'C': 'City', 'N': 'Nature', 'L': 'Landscape', 'S': 'Sunset'}


## Pre-flight check

Before running Qwen, the notebook checks that an image can be found for every selected sample.

This check is performed before model inference so that any missing or incorrectly named images can be identified before the main processing starts.

In [ ]:
preflight_missing = []
for sid in SAMPLE_IDS:
    try:
        find_content_image(sid, category_folders)
    except Exception as e:
        preflight_missing.append((sid, str(e)))

if preflight_missing:
    print(f'PRE-FLIGHT: {len(preflight_missing)}/{len(SAMPLE_IDS)} samples have NO resolvable '
         f'image. Fix these before running Qwen on them, Qwen time is wasted on samples '
         f'that will fail at image discovery regardless.\n')
    _by_prefix = {}
    for sid, err in preflight_missing:
        _by_prefix.setdefault(sid[0], []).append(sid)
    for prefix, ids in sorted(_by_prefix.items()):
        print(f'  {PREFIX_TO_CATEGORY.get(prefix, prefix)}: {len(ids)} missing -- {ids[:5]}'
             f'{"..." if len(ids) > 5 else ""}')
    print(f'\nfirst error in detail:\n  {preflight_missing[0][1]}')
else:
    print(f'PRE-FLIGHT PASSED: all {len(SAMPLE_IDS)} samples have a resolvable image.')

PRE-FLIGHT PASSED: all 200 samples have a resolvable image.


## Prompt and Qwen call

Prompt is used to generate the regional style instructions. The instruction sentence is included directly in the JSON structure shown to Qwen, so the model can follow the required format without having to interpret a separate template.

In [ ]:
INSTRUCTION_TEMPLATE = 'style the {region1} like {style1} and the {region2} like {style2}'
USER_PROMPT_TEMPLATE_VERSION = 'v3_embedded_answer'

SYSTEM_PROMPT = (
    "You are a careful visual grounding assistant. You are given one image and two "
    "pre-specified region labels with a pre-specified artist style assigned to each. "
    "Your job: (1) check whether each named region is actually visible/locatable in this "
    "image, and (2) if both are visible, write a sentence using EXACTLY the words 'style "
    "the', 'like', and 'and the' as shown in the example, substituting in the given region "
    "and style names -- do not paraphrase, do not add or remove regions, do not invent a "
    "different style."
)


def build_user_prompt(r1, r2, s1, s2):
    filled = INSTRUCTION_TEMPLATE.format(region1=r1, style1=s1, region2=r2, style2=s2)
    return (
        f'Look at this image. Two regions are claimed to be present: "{r1}" and "{r2}".\n\n'
        f'Step 1: Is "{r1}" visible in this image? Answer true or false.\n'
        f'Step 2: Is "{r2}" visible in this image? Answer true or false.\n\n'
        f'Step 3: Respond with ONLY this JSON object (no markdown, no other text). '
        f'Copy the "instruction" value EXACTLY character-for-character as written below if '
        f'both regions are visible- it is already written for you, do not modify it, do not '
        f'describe it, just copy it:\n\n'
        f'{{"region1_present": <true or false>, "region2_present": <true or false>, '
        f'"region1_visual_note": "<your note under 15 words>", '
        f'"region2_visual_note": "<your note under 15 words>", '
        f'"instruction": "{filled}" if both true, otherwise "instruction": ""}}'
    )


def prompt_hash():
    return sha256_text(SYSTEM_PROMPT + '-' + USER_PROMPT_TEMPLATE_VERSION)


def call_qwen(image_path, region1, region2, style1, style2):
    from qwen_vl_utils import process_vision_info
    model, processor = load_qwen()
    user_prompt = build_user_prompt(region1, region2, style1, style2)

    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user', 'content': [{'type': 'image', 'image': image_path},
                                     {'type': 'text', 'text': user_prompt}]},
    ]
    chat_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[chat_text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors='pt').to(model.device)

    t0 = time.time()
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=CONFIG['qwen_max_tokens'],
                                 do_sample=False)   # greedy: deterministic
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out_ids)]
    text = processor.batch_decode(trimmed, skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)[0]
    latency = time.time() - t0

    del inputs, out_ids, trimmed
    if HAS_CUDA: torch.cuda.empty_cache()
    gc.collect()
    return text, user_prompt, latency


print('Qwen call defined, prompt version:', USER_PROMPT_TEMPLATE_VERSION)

Qwen call defined, prompt version: v3_embedded_answer


## Schema guard

**Fix 1 -JSON extraction.** Removes markdown code fences and looks for the `{...}` part before parsing the response as JSON.

**Fix 2 -Region matching.** Allows plural forms and partial matches when comparing the region with the design table, rather than requiring an exact match.

These checks only help with formatting and matching. They do not add or change the generated content.

In [ ]:
def extract_instruction_from_raw(raw_text):
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw_text.strip(), flags=re.MULTILINE).strip()
    try:
        obj = json.loads(cleaned)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', cleaned, re.DOTALL)
        if not m:
            return None, f'no JSON object found in raw output: {raw_text[:150]!r}'
        try:
            obj = json.loads(m.group(0))
        except json.JSONDecodeError as e:
            return None, f'raw output is not valid JSON even after extraction: {e}'
    if not isinstance(obj, dict) or 'instruction' not in obj:
        return None, "raw JSON missing required key 'instruction'"
    if not obj.get('region1_present', False) or not obj.get('region2_present', False):
        return None, (f"model reported a region as not present: "
                       f"region1_present={obj.get('region1_present')}, "
                       f"region2_present={obj.get('region2_present')}")
    return obj['instruction'], None


ALLOWED_STYLES = set(CONFIG['allowed_styles'])
STYLE_ALIASES = dict(CONFIG['style_name_aliases'])


def validate_sample(design_row, instruction_text):
    sid = design_row['sample_id']
    r1, r2 = design_row['region1'], design_row['region2']
    exp_s1, exp_s2 = design_row['style1'], design_row['style2']

    if not instruction_text or not instruction_text.strip():
        return 'FAIL', 'empty instruction text', None

    pairs = parse_instruction(instruction_text)
    if len(pairs) != 2:
        return 'FAIL', f'expected 2 (region, style) pairs, regex found {len(pairs)}', None
    by_region = {p['region']: p['style'] for p in pairs}

    def _find(target):
        if target in by_region: return by_region[target]
        for k, v in by_region.items():
            if k.rstrip('s') == target.rstrip('s') or target in k or k in target:
                return v
        return None

    got_s1_raw, got_s2_raw = _find(r1), _find(r2)
    if got_s1_raw is None or got_s2_raw is None:
        return ('FAIL', f'could not match design regions {(r1, r2)} against parsed regions '
                        f'{sorted(by_region.keys())}', None)

    got_s1 = STYLE_ALIASES.get(got_s1_raw, got_s1_raw)
    got_s2 = STYLE_ALIASES.get(got_s2_raw, got_s2_raw)
    for style in (got_s1, got_s2):
        if style not in ALLOWED_STYLES:
            return 'FAIL', f'style "{style}" not in allowed vocabulary', None
    if got_s1 != exp_s1 or got_s2 != exp_s2:
        return ('FAIL', f'style mismatch: expected ({exp_s1}, {exp_s2}), got '
                        f'({got_s1}, {got_s2})', None)

    return 'PASS', None, {
        'id': sid, 'regions': [r1, r2], 'instruction': instruction_text.strip(),
        'style1_resolved': got_s1, 'style2_resolved': got_s2,
    }

print('schema guard ready (2 fixes applied)')

schema guard ready (2 fixes applied)


---
## One-sample smoke test

In [ ]:
_smoke_id = next((s for s in SAMPLE_IDS if not any(s == m[0] for m in preflight_missing)),
                 SAMPLE_IDS[0])
row = DESIGN_BY_ID[_smoke_id]
image_path = find_content_image(_smoke_id, category_folders)
raw, prompt, latency = call_qwen(image_path, row['region1'], row['region2'],
                                  row['style1'], row['style2'])
print(raw)

_instr, _reason = extract_instruction_from_raw(raw)
assert _instr is not None, f'smoke test FAILED at extraction: {_reason}'
_status, _reason, _ = validate_sample(row, _instr)
assert _status == 'PASS', f'smoke test FAILED at validation: {_reason}'
print(f'\nsmoke test PASSED on {_smoke_id}: "{_instr}"')

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

loaded Qwen/Qwen2-VL-2B-Instruct, 4bit=False
VRAM free after load: 11.05 GB
{"region1_present": true, "region2_present": true, "region1_visual_note": "The sky is clear and blue.", "region2_visual_note": "The buildings are modern and reflective.", "instruction": "style the sky like Cezanne and the buildings like Van Gogh"}

smoke test PASSED on C001: "style the sky like Cezanne and the buildings like Van Gogh"


---
## Main loop Checkpointing and resuming

The notebook saves progress every 10 samples. The checkpoint is first written to a temporary file and then renamed to the main checkpoint file.

If the Colab session stops, the notebook can use the saved checkpoint when it is run again, so samples that have already been processed do not need to be processed again.


In [ ]:
def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH) as f:
            return json.load(f)
    return []


def save_checkpoint(records):
    tmp = CHECKPOINT_PATH + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(records, f)
    os.replace(tmp, CHECKPOINT_PATH)


records = load_checkpoint()
done_ids = {r['sample_id'] for r in records}
if done_ids:
    print(f'RESUMING from checkpoint: {len(done_ids)} sample(s) already processed.')

remaining = [sid for sid in SAMPLE_IDS if sid not in done_ids]
print(f'{len(remaining)} sample(s) remaining this session '
     f'(estimated {len(remaining) * 5.2 / 60:.0f} min at ~5.2s/sample)')

_t0 = time.time()
for i, sid in enumerate(remaining, 1):
    row = DESIGN_BY_ID[sid]
    rec = {'sample_id': sid, 'category': PREFIX_TO_CATEGORY[sid[0].upper()],
          'design_region1': row['region1'], 'design_region2': row['region2'],
          'design_style1': row['style1'], 'design_style2': row['style2'],
          'run_id': RUN_ID, 'qwen_model': CONFIG['qwen_model']}
    try:
        image_path = find_content_image(sid, category_folders)
        rec['image_path'] = image_path
        rec['image_sha256'] = sha256_file(image_path)
    except Exception as e:
        rec.update({'raw_qwen_output': None, 'validation_status': 'FAIL',
                    'failure_reason': f'image discovery failed: {e}', 'validated_output': None})
        records.append(rec)
        if i % 10 == 0: save_checkpoint(records)
        continue

    try:
        raw_text, user_prompt, latency = call_qwen(image_path, row['region1'], row['region2'],
                                                    row['style1'], row['style2'])
    except Exception as e:
        rec.update({'raw_qwen_output': None, 'validation_status': 'FAIL',
                    'failure_reason': f'qwen call failed: {type(e).__name__}: {e}',
                    'validated_output': None})
        records.append(rec)
        if i % 10 == 0: save_checkpoint(records)
        continue

    with open(f'{RAW_DIR}/{sid}.json', 'w') as f:
        json.dump({'sample_id': sid, 'raw_qwen_output': raw_text, 'user_prompt': user_prompt}, f)

    rec.update({'raw_qwen_output': raw_text, 'qwen_latency_s': latency, 'prompt_hash': prompt_hash()})
    instr, extract_reason = extract_instruction_from_raw(raw_text)
    if instr is None:
        rec.update({'validation_status': 'FAIL', 'failure_reason': extract_reason,
                    'validated_output': None})
    else:
        status, reason, validated = validate_sample(row, instr)
        rec.update({'validation_status': status, 'failure_reason': reason,
                    'validated_output': validated})
    records.append(rec)

    if i % 10 == 0 or i == len(remaining):
        save_checkpoint(records)
        n_pass = sum(1 for r in records if r['validation_status'] == 'PASS')
        el = time.time() - _t0
        print(f'{i}/{len(remaining)} this session | {n_pass}/{len(records)} PASS total | '
              f'{el:.0f}s elapsed | checkpoint saved to Drive')

n_pass = sum(1 for r in records if r['validation_status'] == 'PASS')
print(f'\nDONE: {n_pass}/{len(records)} PASS ({n_pass/len(records):.0%}) '
     f'out of {len(records)}/{len(SAMPLE_IDS)} samples processed')
if len(records) < len(SAMPLE_IDS):
    print(f'NOTE: {len(SAMPLE_IDS) - len(records)} sample(s) not yet processed -- re-run '
         f'this cell to continue from the checkpoint.')
unload_qwen()

200 sample(s) remaining this session (estimated 17 min at ~5.2s/sample)
10/200 this session | 10/10 PASS total | 68s elapsed | checkpoint saved to Drive
20/200 this session | 20/20 PASS total | 132s elapsed | checkpoint saved to Drive
30/200 this session | 30/30 PASS total | 191s elapsed | checkpoint saved to Drive
40/200 this session | 40/40 PASS total | 252s elapsed | checkpoint saved to Drive
50/200 this session | 50/50 PASS total | 307s elapsed | checkpoint saved to Drive
60/200 this session | 60/60 PASS total | 364s elapsed | checkpoint saved to Drive
70/200 this session | 70/70 PASS total | 426s elapsed | checkpoint saved to Drive
80/200 this session | 80/80 PASS total | 494s elapsed | checkpoint saved to Drive
90/200 this session | 90/90 PASS total | 552s elapsed | checkpoint saved to Drive
100/200 this session | 100/100 PASS total | 615s elapsed | checkpoint saved to Drive
110/200 this session | 110/110 PASS total | 677s elapsed | checkpoint saved to Drive
120/200 this session 

---
## Failure diagnostics

In [ ]:
import pandas as pd

fails = [r for r in records if r['validation_status'] == 'FAIL']
if fails:
    reasons = pd.Series([r['failure_reason'] for r in fails])
    print(f'{len(fails)}/{len(records)} failed\n')
    print(reasons.value_counts().head(15).to_string())

    print('\nfailure rate by category:')
    fail_df = pd.DataFrame([{'category': r['category'],
                             'failed': r['validation_status'] == 'FAIL'} for r in records])
    print(fail_df.groupby('category').failed.agg(['sum', 'count', 'mean']).to_string())

    print('\none example per distinct reason category:')
    seen = set()
    for r in fails:
        key = (r['failure_reason'] or '')[:30]
        if key in seen: continue
        seen.add(key)
        print(f"\n  {r['sample_id']}: {r['failure_reason']}")
        if r.get('raw_qwen_output'):
            print(f"    raw: {r['raw_qwen_output'][:200]!r}")
else:
    print('no failures')

no failures


---
## Freeze the manifest and save

In [ ]:
MANIFEST_PATH = f'{MANIFEST_DIR}/qwen_manifest.json'
INSTRUCTIONS_PATH = f'{MANIFEST_DIR}/instructions_from_qwen.json'
PROVENANCE_PATH = f'{MANIFEST_DIR}/qwen_run_provenance.json'

with open(MANIFEST_PATH, 'w') as f:
    json.dump(records, f, indent=2, sort_keys=True)
manifest_sha = sha256_file(MANIFEST_PATH)

passing = [{'id': r['validated_output']['id'], 'regions': r['validated_output']['regions'],
           'instruction': r['validated_output']['instruction']}
          for r in records if r['validation_status'] == 'PASS']
with open(INSTRUCTIONS_PATH, 'w') as f:
    json.dump(passing, f, indent=2)

provenance = {
    'run_id': RUN_ID, 'qwen_model': CONFIG['qwen_model'],
    'qwen_load_in_4bit': CONFIG['qwen_load_in_4bit'],
    'prompt_version': USER_PROMPT_TEMPLATE_VERSION, 'prompt_hash': prompt_hash(),
    'n_samples': len(records),
    'n_pass': sum(1 for r in records if r['validation_status'] == 'PASS'),
    'n_fail': sum(1 for r in records if r['validation_status'] == 'FAIL'),
    'manifest_path': MANIFEST_PATH, 'manifest_sha256': manifest_sha,
    'instructions_path': INSTRUCTIONS_PATH,
}
with open(PROVENANCE_PATH, 'w') as f:
    json.dump(provenance, f, indent=2)

print(f'manifest: {MANIFEST_PATH}  (sha256 {manifest_sha[:16]})')
print(f'instructions: {INSTRUCTIONS_PATH}  ({len(passing)} passing samples)')
print(f'provenance: {PROVENANCE_PATH}')
print(f'\n{len(passing)}/{len(records)} samples ready for the downstream notebook')
for p in (MANIFEST_PATH, INSTRUCTIONS_PATH, PROVENANCE_PATH):
    assert os.path.exists(p), f'{p} missing after write'
print('\nconfirmed: all output files are under a Drive-backed path')

if 'google.colab' in sys.modules:
    from google.colab import files as colab_files
    for p in (MANIFEST_PATH, INSTRUCTIONS_PATH, PROVENANCE_PATH):
        colab_files.download(p)
    print('browser download triggered for all 3 files')

manifest        : /content/drive/MyDrive/qwen_grounding/manifest/qwen_manifest.json  (sha256 b454a1590b65a6b4)
instructions    : /content/drive/MyDrive/qwen_grounding/manifest/instructions_from_qwen.json  (200 passing samples)
provenance      : /content/drive/MyDrive/qwen_grounding/manifest/qwen_run_provenance.json

200/200 samples ready for the downstream notebook

confirmed: all output files are under a Drive-backed path


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

browser download triggered for all 3 files


In [ ]:
#Verify Qwen's output against instructions_200.json
import json

with open(CONFIG['existing_instructions_path']) as f:
    original = json.load(f)
original_by_id = {s['id']: s for s in original}

with open(INSTRUCTIONS_PATH) as f:
    qwen_out = json.load(f)
qwen_by_id = {s['id']: s for s in qwen_out}

print(f'original instructions_200.json : {len(original)} samples')
print(f'qwen-produced instructions: {len(qwen_out)} samples')
print(f'coverage: {len(qwen_out)}/{len(original)} '
      f'({len(qwen_out)/len(original):.0%})\n')

# 1. check whether original IDs are missing from Qwen's output
missing_ids = sorted(set(original_by_id) - set(qwen_by_id))
if missing_ids:
    print(f'MISSING from Qwen output ({len(missing_ids)}): {missing_ids[:10]}'
         f'{"..." if len(missing_ids) > 10 else ""}')
else:
    print('no missing sample IDs')

# 2. For each sample where Qwen produced an output, check that the regions and
#    resolved styles match the original. The wording can be different, but the
#    region/style pairing needs to stay the same because it determines the
#    images and conditioning used downstream.
mismatches = []
for sid, q in qwen_by_id.items():
    orig = original_by_id.get(sid)
    if orig is None:
        mismatches.append((sid, 'not in original instructions_200.json at all'))
        continue
    if q['regions'] != orig['regions']:
        mismatches.append((sid, f"regions differ: qwen={q['regions']} vs "
                                f"original={orig['regions']}"))
        continue
    # Re-parse both instructions using the same downstream regex and compare the
# resolved styles. The wording can differ, but the style assignment should stay the same.
    q_pairs = {p['region']: p['style'] for p in parse_instruction(q['instruction'])}
    o_pairs = {p['region']: p['style'] for p in parse_instruction(orig['instruction'])}
    q_styles = tuple(CONFIG['style_name_aliases'].get(q_pairs.get(r), q_pairs.get(r))
                     for r in q['regions'])
    o_styles = tuple(CONFIG['style_name_aliases'].get(o_pairs.get(r), o_pairs.get(r))
                     for r in orig['regions'])
    if q_styles != o_styles:
        mismatches.append((sid, f'style assignment differs: qwen={q_styles} vs '
                                f'original={o_styles}'))

if mismatches:
    print(f'\nSTYLE/REGION MISMATCHES ({len(mismatches)}):')
    for sid, reason in mismatches[:15]:
        print(f'{sid}: {reason}')
    if len(mismatches) > 15:
        print(f'  ... and {len(mismatches) - 15} more')
else:
    print(f'\nall {len(qwen_by_id)} produced samples match the original design exactly '
         f'(same regions, same resolved styles)')

# 3. Category coverage :40/40 per category
import pandas as pd
cov = pd.DataFrame([{'category': PREFIX_TO_CATEGORY[sid[0].upper()], 'sample_id': sid}
                    for sid in qwen_by_id])
counts = cov.category.value_counts().reindex(CONFIG['categories'], fill_value=0)
print('\ncategory coverage (need 40/40 each for the downstream 200-sample benchmark):')
print(counts.to_string())
short = counts[counts < 40]

# 4. Every remaining region/style value must be in the downstream notebook's known
#    vocabulary, or generation will fail later with a much less clear error.
vocab_problems = []
for sid, q in qwen_by_id.items():
    pairs = {p['region']: p['style'] for p in parse_instruction(q['instruction'])}
    for r in q['regions']:
        style = CONFIG['style_name_aliases'].get(pairs.get(r), pairs.get(r))
        if style not in CONFIG['allowed_styles']:
            vocab_problems.append((sid, r, style))
if vocab_problems:
    print(f'\nSTYLE VOCABULARY PROBLEMS ({len(vocab_problems)}): {vocab_problems[:10]}')
print('\n' + '=' * 60)
ready = (not missing_ids and not mismatches and short.empty and not vocab_problems)
if ready:
    print(f'{INSTRUCTIONS_PATH}')
else:
    print('NOT READY. Issues above must be resolved first:')
    if missing_ids: print(f'- {len(missing_ids)} sample(s) missing entirely')
    if mismatches:print(f'- {len(mismatches)} sample(s) with mismatched regions/styles')
    if not short.empty:print(f'- categories under 40: {short.to_dict()}')
    if vocab_problems:print(f'- {len(vocab_problems)} style vocabulary problem(s)')
print('=' * 60)

original instructions_200.json : 200 samples
qwen-produced instructions      : 200 samples
coverage                        : 200/200 (100%)

no missing sample IDs

all 200 produced samples match the original design exactly (same regions, same resolved styles)

category coverage (need 40/40 each for the downstream 200-sample benchmark):
category
City         40
Landscape    40
Nature       40
Sunset       40
Winter       40

  /content/drive/MyDrive/qwen_grounding/manifest/instructions_from_qwen.json


In [ ]:
# Summary as a dataframe
import pandas as pd

summary_df = pd.DataFrame([
    {'check': 'Total samples (original)','value': len(original),'status': 'OK'},
    {'check': 'Total samples (Qwen)','value': len(qwen_by_id),'status': 'OK'},
    {'check': 'Coverage','value': f'{len(qwen_by_id)}/{len(original)} (100%)', 'status': 'OK'},
    {'check': 'Missing sample IDs','value': len(missing_ids),'status': 'OK' if not missing_ids else 'FAIL'},
    {'check': 'Region/style mismatches','value': len(mismatches),'status': 'OK' if not mismatches else 'FAIL'},
    {'check': 'Categories below 40','value': len(short),'status': 'OK' if short.empty else 'FAIL'},
    {'check': 'Style vocabulary problems','value': len(vocab_problems),'status': 'OK' if not vocab_problems else 'FAIL'},
    {'check': 'READY for downstream notebook','value': ready,'status': 'YES' if ready else 'NO'},
])
print(summary_df.to_string(index=False))

                        check          value status
     Total samples (original)            200     OK
         Total samples (Qwen)            200     OK
                     Coverage 200/200 (100%)     OK
           Missing sample IDs              0     OK
      Region/style mismatches              0     OK
          Categories below 40              0     OK
    Style vocabulary problems              0     OK
READY for downstream notebook           True    YES


In [ ]:
category_df = counts.reset_index()
category_df.columns = ['category', 'n_samples']
category_df['expected'] = 40
category_df['status'] = category_df.n_samples.eq(40).map({True: 'OK', False: 'SHORT'})
print(category_df.to_string(index=False))

 category  n_samples  expected status
     City         40        40     OK
Landscape         40        40     OK
   Nature         40        40     OK
   Sunset         40        40     OK
   Winter         40        40     OK


In [ ]:
from IPython.display import display

print('VERIFICATION SUMMARY')
display(summary_df)
print('\nCATEGORY COVERAGE')
display(category_df)

VERIFICATION SUMMARY


,check,value,status
0,Total samples (original),200,OK
1,Total samples (Qwen),200,OK
2,Coverage,200/200 (100%),OK
3,Missing sample IDs,0,OK
4,Region/style mismatches,0,OK
5,Categories below 40,0,OK
6,Style vocabulary problems,0,OK
7,READY for downstream notebook,True,YES



CATEGORY COVERAGE


,category,n_samples,expected,status
0,City,40,40,OK
1,Landscape,40,40,OK
2,Nature,40,40,OK
3,Sunset,40,40,OK
4,Winter,40,40,OK




```python
# In the downstream notebook's CONFIG:
CONFIG['instructions_path'] = '/content/drive/MyDrive/qwen_grounding/manifest/instructions_from_qwen.json'
```